# 🏗️ FinTrack Architecture Blueprint

**Generated:** 2026-07-26 | **Version:** 1.0 | **Blueprint ID:** FT-ARCH-001

---

## 📋 Quick Navigation

| # | Section | Description |
|---|---------|-------------|
| 1 | [Architecture Detection](#1) | Technology stack & pattern analysis |
| 2 | [Architectural Overview](#2) | High-level design philosophy |
| 3 | [Architecture Visualization](#3) | C4 diagrams (Mermaid) |
| 4 | [Core Components](#4) | Module-by-module deep dive |
| 5 | [Layers & Dependencies](#5) | Dependency rules & layer map |
| 6 | [Data Architecture](#6) | MongoDB schema & access patterns |
| 7 | [Cross-Cutting Concerns](#7) | Auth, logging, validation, config |
| 8 | [Service Communication](#8) | HTTP, RabbitMQ, events |
| 9 | [.NET-Specific Patterns](#9) | Controllers, DI, middleware |
| 10 | [Implementation Patterns](#10) | Concrete code templates |
| 11 | [Testing Architecture](#11) | xUnit strategy |
| 12 | [Deployment Architecture](#12) | Docker, compose, topology |
| 13 | [Extension & Evolution](#13) | Adding features, modules |
| 14 | [Code Examples](#14) | Extracted patterns from codebase |
| 15 | [ADR — Decision Records](#15) | Why we built it this way |
| 16 | [Governance](#16) | Rules for maintaining consistency |
| 17 | [New Dev Blueprint](#17) | Templates & pitfalls for new features |


## 1. Architecture Detection & Analysis {#1}

### Technology Stack

| Category | Technology | Version | Where |
|----------|-----------|---------|-------|
| **Runtime** | .NET | 10.0 | All `.csproj` files |
| **Language** | C# 14 | — | `.cs` files |
| **Web Framework** | ASP.NET Core (Controllers) | 10.0 | `FinTrack.Api` |
| **Database** | MongoDB + Mongo.Entities | 7.0 / 24.1 | `BuildingBlocks` |
| **Messaging** | MassTransit + RabbitMQ | 8.4 / 3.x | `FinTrack.Api`, `FinTrack.Host` |
| **Auth** | JWT Bearer + BCrypt | — | `Modules.Users` |
| **Validation** | FluentValidation | 12.1 | All modules |
| **Mediator** | MediatR | 14.1 | All modules |
| **Testing** | xUnit + FluentAssertions + NSubstitute | 2.9 / 8.2 / 5.3 | `tests/` |
| **Infrastructure** | Docker Compose | — | `docker-compose.yml` |

### Detected Architectural Pattern

**Primary:** Modular Monolith + Vertical Slice Architecture (VSA)

**Evidence:**
- 6 bounded-context modules as class libraries (`FinTrack.Modules.*`)
- No cross-module project references — only `BuildingBlocks` + `Contracts`
- Each module has `Features/{UseCase}/` folders with Command + Handler + Validator + Controller
- `internal` visibility on handlers; only `DependencyInjection.cs` is `public`
- Two deployable hosts (`Api` + `Host`) share the same module assemblies

**Secondary Patterns:**
- **CQRS** — Commands mutate, Queries read (via MediatR `IRequest<T>`)
- **Event-Driven** — Cross-module communication via MassTransit/RabbitMQ integration events
- **Result Pattern** — `BuildingBlocks.Result<T>` for operation outcomes


## 2. Architectural Overview {#2}

### Design Philosophy

FinTrack follows **Modular Monolith** principles: all business logic runs in-process within two deployable hosts (`Api` + `Host`), but modules are isolated by convention — not by network boundaries.

**Guiding Principles:**
1. **Module = Bounded Context** — Each module owns its data and logic. No shared entity types across modules.
2. **No Cross-Module References** — Communication only via integration events (RabbitMQ) or Contracts DTOs.
3. **Auth is a Hard Gate** — Global `[Authorize]` fallback. Only `Register`, `Login`, `RefreshToken` are anonymous.
4. **User-Scoped Data** — Every document stores `UserId`. All queries filter by `ICurrentUser`.
5. **Internal by Default** — Handlers, validators are `internal`. Only DI entrypoints and controllers are `public`.
6. **Vertical Slices** — One feature = one folder = one use case. No shared "Application Service" layer.

### Module Prominence Order

```
Dashboard → Transactions → Categories → Budgets → Accounts → Users
```

> Dashboard is the top product surface but is **read-only**. Users owns auth and ships first.

### Hybrid Pattern Adaptation

| Standard VSA | FinTrack Adaptation |
|-------------|-------------------|
| Feature folders with endpoint + handler + validator | ✅ Plus separate `Controllers/` folder per module |
| MediatR for command dispatch | ✅ All features use `ISender` |
| Domain folder per module | ✅ `Domain/` with Mongo.Entities entities |
| No cross-module references | ✅ Enforced via `.csproj` (no module→module ProjectReference) |


## 3. Architecture Visualization {#3}

### C4 — Context Diagram

```mermaid
graph TB
    User["👤 User (React SPA)"]
    
    subgraph FinTrack["FinTrack System"]
        Api["🔵 FinTrack.Api
        HTTP + JWT + Swagger"]
        Host["🟢 FinTrack.Host
        MassTransit Workers"]
    end
    
    Mongo[("🍃 MongoDB
    FinTrackDb")]
    Rabbit[("🐰 RabbitMQ
    Events")]
    
    User -->|"HTTPS :5171"| Api
    Api -->|"Read/Write"| Mongo
    Api -->|"Publish Events"| Rabbit
    Host -->|"Consume Events"| Rabbit
    Host -->|"Read/Write"| Mongo
    
    style Api fill:#4A90D9,color:#fff
    style Host fill:#27AE60,color:#fff
    style Mongo fill:#3FA037,color:#fff
    style Rabbit fill:#FF6600,color:#fff
```

### C4 — Container Diagram

```mermaid
graph TB
    subgraph Api["FinTrack.Api"]
        Auth["JWT Middleware"]
        CORS["CORS Policy"]
        Swagger["Swagger UI"]
        Controllers["Controllers"]
        MediatR1["MediatR Pipeline"]
        MT1["MassTransit (Publish only)"]
    end
    
    subgraph Host["FinTrack.Host"]
        MediatR2["MediatR"]
        MT2["MassTransit (Consume)"]
        Consumers["Event Consumers"]
    end
    
    subgraph Modules["Shared Modules (6)"]
        Dashboard["Dashboard"]
        Transactions["Transactions"]
        Categories["Categories"]
        Budgets["Budgets"]
        Accounts["Accounts"]
        Users["Users"]
    end
    
    subgraph Shared["Shared"]
        BB["BuildingBlocks"]
        Contracts["Contracts"]
    end
    
    Controllers --> MediatR1
    MediatR1 --> Modules
    MediatR2 --> Modules
    Consumers --> Modules
    Modules --> BB
    Modules --> Contracts
    
    style Api fill:#4A90D9,color:#fff
    style Host fill:#27AE60,color:#fff
    style Modules fill:#8E44AD,color:#fff
    style Shared fill:#F39C12,color:#000
```

### C4 — Component Diagram (Transactions Module)

```mermaid
graph LR
    subgraph Controller["Controller Layer"]
        TC["TransactionsController
        [HttpPost] CreateTransaction()"]
    end
    
    subgraph Feature["Feature Slice"]
        Cmd["CreateTransactionCommand
        record (IRequest)"]
        Handler["CreateTransactionHandler
        IRequestHandler"]
        Val["CreateTransactionValidator
        AbstractValidator"]
    end
    
    subgraph Domain["Domain"]
        Entity["Transaction : AuditableEntity
        [Collection('transactions')]"]
    end
    
    subgraph Infra["Infrastructure"]
        Mongo["Mongo.Entities
        SaveAsync()"]
        MT["IPublishEndpoint
        MassTransit"]
    end
    
    TC -->|"ISender.Send()"| Cmd
    Cmd --> Handler
    Handler --> Val
    Handler --> Entity
    Handler --> Mongo
    Handler --> MT
    
    style Controller fill:#E74C3C,color:#fff
    style Feature fill:#3498DB,color:#fff
    style Domain fill:#2ECC71,color:#fff
    style Infra fill:#95A5A6,color:#000
```


## 4. Core Architectural Components {#4}

### 4.1 FinTrack.Api — HTTP Host

| Aspect | Detail |
|--------|--------|
| **Purpose** | Entry point for all HTTP traffic. No business logic. |
| **Framework** | ASP.NET Core 10 — WebApplication host |
| **Middleware** | HTTPS → CORS → Authentication → Authorization → Controllers |
| **Auth** | JWT Bearer with global `[Authorize]` fallback |
| **Messaging** | MassTransit — **publish only** (no long-running consumers) |
| **Controllers** | Auto-discovered via `AddControllers()` + `MapControllers()` |
| **Swagger** | Available in Development at `/swagger` |

**Key File:** `src/FinTrack.Api/Program.cs`

```csharp
// Composition root — wires everything together
builder.Services.AddAuthentication().AddJwtBearer(...);
builder.Services.AddAuthorization(o => o.FallbackPolicy = o.DefaultPolicy);
builder.Services.AddMediatR(cfg => { /* all 6 module assemblies */ });
builder.Services.AddMassTransit(cfg => cfg.UsingRabbitMq(...));
builder.Services.AddDashboardModule();
builder.Services.AddTransactionsModule();
// ... remaining modules ...
builder.Services.AddControllers();    // ← Controllers, not Minimal APIs

app.MapControllers();                  // ← Auto-discovers all [ApiController] classes
```

### 4.2 FinTrack.Host — Background Worker Host

| Aspect | Detail |
|--------|--------|
| **Purpose** | MassTransit consumer host — reacts to integration events |
| **Framework** | `Host.CreateApplicationBuilder()` — no HTTP |
| **ICurrentUser** | `SystemCurrentUser` — `UserId = null` (trusts event payloads) |
| **Messaging** | MassTransit — **consume only** + `ConfigureEndpoints` |
| **Consumers** | Registered per module: Dashboard, Budgets, Accounts, Categories |

**Key File:** `src/FinTrack.Host/Program.cs`

### 4.3 Module Structure (all 6 follow this pattern)

```
FinTrack.Modules.{Name}/
├── DependencyInjection.cs       ← Public: Add{Name}Module()
├── Controllers/
│   └── {Name}Controller.cs      ← Public: [ApiController] + [Authorize]
├── Domain/
│   └── {Entity}.cs              ← Internal: AuditableEntity subclass
└── Features/
    └── {UseCase}/
        ├── {UseCase}Command.cs   ← record : IRequest<T>
        ├── {UseCase}Handler.cs   ← internal sealed : IRequestHandler
        └── {UseCase}Validator.cs ← internal sealed : AbstractValidator
```

| Module | Entity | Collection | Write Features | Read Features |
|--------|--------|-----------|---------------|---------------|
| **Dashboard** | `DashboardSnapshot` | `dashboardSnapshots` | — (projection-only) | `GetDashboardSummary` |
| **Transactions** | `Transaction` | `transactions` | `CreateTransaction` | — |
| **Categories** | `Category` | `categories` | `CreateCategory` | — |
| **Users** | `User`, `RefreshToken` | `users`, `refreshTokens` | `Register`, `Login`, `RefreshToken` | — |
| **Budgets** | `Budget` | `budgets` | `CreateBudget` | — |
| **Accounts** | `Account` | `accounts` | `CreateAccount` | — |

### 4.4 BuildingBlocks — Cross-Cutting Primitives

| Type | File | Purpose |
|------|------|---------|
| `ICurrentUser` | `ICurrentUser.cs` | Abstraction over authenticated user |
| `AuditableEntity` | `AuditableEntity.cs` | Base class wrapping `Mongo.Entities.Entity` |
| `Result<T>` | `Result.cs` | Success/failure pattern |
| `MongoInitializer` | `MongoInitializer.cs` | `DB.InitAsync()` bootstrap |
| `LoggingBehavior` | `Behaviors/LoggingBehavior.cs` | MediatR pipeline — log every request |
| `ValidationBehavior` | `Behaviors/ValidationBehavior.cs` | MediatR pipeline — validate every request |

### 4.5 Contracts — Integration Events

| File | Events |
|------|--------|
| `Users/UserRegisteredEvent.cs` | `UserRegisteredEvent` |
| `Transactions/TransactionEvents.cs` | `TransactionCreatedEvent`, `TransactionUpdatedEvent` |
| `Categories/CategoryEvents.cs` | `CategoryCreatedEvent` |
| `Budgets/BudgetEvents.cs` | `BudgetExceededEvent` |
| `Accounts/AccountEvents.cs` | `AccountCreatedEvent`, `AccountClosedEvent` |

> All events are immutable `record` types. Past-tense naming. No Mongo entity types.


## 5. Architectural Layers & Dependencies {#5}

### Dependency Graph

```
┌─────────────────────────────────────────────┐
│              FinTrack.Api                    │
│         (Composition Root)                   │
└──────┬──────┬──────┬──────┬──────┬──────────┘
       │      │      │      │      │
       ▼      ▼      ▼      ▼      ▼
  Dashboard Txns   Cats   Budgets  Accts  Users
       │      │      │      │      │      │
       └──────┴──────┴──────┴──────┴──────┘
                     │
              ┌──────┴──────┐
              ▼              ▼
       BuildingBlocks    Contracts
              │
              ▼
       Mongo.Entities + MassTransit
```

### Dependency Rules

| Rule | Enforcement |
|------|------------|
| **Modules never reference each other** | `.csproj` — no `ProjectReference` between `FinTrack.Modules.*` |
| **Modules depend on BuildingBlocks + Contracts** | `.csproj` — all modules reference these two |
| **Api depends on all modules** | `.csproj` — `ProjectReference` to all 6 modules |
| **Host depends on all modules** | `.csproj` — same as Api |
| **Contracts has zero dependencies** | `.csproj` — no PackageReference or ProjectReference |
| **BuildingBlocks has framework deps only** | Mongo.Entities, MediatR, FluentValidation, JWT |

### Layer Violation Check

✅ **No circular dependencies detected.** Module graph is a DAG (Directed Acyclic Graph).

✅ **No module-to-module references.** Cross-module communication uses only:
- Integration events (`Contracts/*Event.cs`) via RabbitMQ
- String-based foreign keys (`UserId`, `AccountId`, `CategoryId`)

### Dependency Injection Flow

```
Program.cs
  ├── builder.Services.AddScoped<ICurrentUser, HttpContextCurrentUser>()
  ├── builder.Services.AddMediatR(6 assemblies) → discovers all IRequestHandler<T>
  ├── builder.Services.AddValidatorsFromAssemblies(5 assemblies) → discovers all AbstractValidator<T>
  ├── builder.Services.AddMassTransit(...) → IPublishEndpoint available to handlers
  └── builder.Services.Add*Module() → per-module registrations (JwtOptions, etc.)
```


## 6. Data Architecture {#6}

### MongoDB Database: `FinTrackDb`

| Collection | Owning Module | Document Type | Key Fields |
|-----------|--------------|---------------|------------|
| `users` | Users | `User` | `Email`, `PasswordHash` |
| `refreshTokens` | Users | `RefreshToken` | `UserId`, `Token`, `ExpiresAt`, `IsRevoked` |
| `transactions` | Transactions | `Transaction` | `UserId`, `AccountId`, `CategoryId`, `Title`, `Amount`, `Type` |
| `categories` | Categories | `Category` | `UserId`, `Name`, `Type` |
| `accounts` | Accounts | `Account` | `UserId`, `Name`, `Type`, `Balance`, `IsClosed` |
| `budgets` | Budgets | `Budget` | `UserId`, `CategoryId`, `Limit`, `CurrentSpend`, `Period` |
| `dashboardSnapshots` | Dashboard | `DashboardSnapshot` | `UserId`, `TotalBalance`, `TotalIncome`, `TotalExpense` |

### Entity Inheritance Chain

```
MongoDB.Entities.Entity          ← Framework base (ID, CreatedOn, ModifiedOn)
    └── FinTrack.BuildingBlocks.AuditableEntity   ← Adds: CreatedBy, CreateDate, LastUpdatedBy, LastUpdateDate, TimeZoneOffsetInMinutes
        └── FinTrack.Modules.{X}.Domain.{Entity}  ← Module-specific fields
```

### Data Access Patterns

| Pattern | Implementation | Example |
|---------|---------------|---------|
| **Insert/Update** | `entity.SaveAsync()` | `CreateTransactionHandler` — `await transaction.SaveAsync()` |
| **Find One** | `DB.Find<T>().Match(...).ExecuteFirstAsync()` | `LoginHandler` — find user by email |
| **Find Many** | `DB.Find<T>().Match(...).ExecuteAsync()` | Future: list transactions |
| **Bulk Update** | `DB.Update<T>().Match(...).Modify(...).ExecuteAsync()` | `LoginHandler` — revoke all refresh tokens |

### Data Scoping Convention

```csharp
// Every query MUST filter by UserId
var user = await DB.Find<User>()
    .Match(u => u.Email == email)    // ← business filter
    .ExecuteFirstAsync();            // UserId scoping is implicit via the entity itself
```

> The `ICurrentUser.UserId` pattern ensures handlers only access the current user's data. No cross-user data leakage.

### Cross-Module References

Foreign keys are **string IDs only** — never object references:

```csharp
public class Transaction : AuditableEntity
{
    public string UserId { get; set; }       // ← string, not User object
    public string AccountId { get; set; }    // ← string, not Account object
    public string CategoryId { get; set; }   // ← string, not Category object
}
```

Existence validation is done at the handler level (caller supplies valid IDs) with eventual consistency — not via cross-module DB queries.


## 7. Cross-Cutting Concerns {#7}

### 7.1 Authentication & Authorization

```
┌─────────────────────────────────────────────────────┐
│                  REQUEST FLOW                       │
│                                                     │
│  Client ──► [Auth Middleware] ──► [Authz Middleware] ──► Controller ──► Handler
│                │                      │                    │            │
│                │ Validate JWT         │ Check policy        │ [Authorize]│ ICurrentUser
│                │ (SigningKey,         │ (FallbackPolicy     │ attribute  │ .UserId
│                │  Issuer, Audience)   │  = DefaultPolicy)   │            │
│                                                     │
│  Anonymous endpoints: Register, Login, RefreshToken │
│  └── [AllowAnonymous] bypasses both middlewares     │
└─────────────────────────────────────────────────────┘
```

**Implementation chain:**

| Step | File | What |
|------|------|------|
| JWT config | `Program.cs` | `AddAuthentication().AddJwtBearer(...)` |
| Fallback policy | `Program.cs` | `options.FallbackPolicy = options.DefaultPolicy` |
| Controller attribute | `*Controller.cs` | `[Authorize]` on class, `[AllowAnonymous]` on auth actions |
| User identity | `HttpContextCurrentUser.cs` | Reads `userId` and `email` claims |
| Handler check | `*Handler.cs` | `_currentUser.UserId ?? throw UnauthorizedAccessException` |
| Password hashing | `RegisterHandler.cs` / `LoginHandler.cs` | `BCrypt.HashPassword()` / `BCrypt.Verify()` |
| Token generation | `JwtTokenGenerator.cs` | HMAC-SHA256, 15-min access, 7-day refresh |
| Refresh rotation | `RefreshTokenHandler.cs` | Revoke old token → issue new pair |

### 7.2 Error Handling

| Pattern | Implementation |
|---------|---------------|
| **Validation errors** | `ValidationBehavior` throws `FluentValidation.ValidationException` → MVC returns 400 |
| **Auth errors** | `UnauthorizedAccessException` → MVC returns 401 |
| **Business rule errors** | `InvalidOperationException` (e.g., duplicate email) → MVC returns 400 |
| **Handler logging** | Every handler logs at `Information` level via `ILogger<T>` |
| **Pipeline logging** | `LoggingBehavior` logs every MediatR request/response |

### 7.3 Logging

- **Standard:** Microsoft.Extensions.Logging via `ILogger<T>` injected into handlers
- **Pipeline:** `LoggingBehavior<TRequest, TResponse>` wraps every MediatR handler automatically
- **Levels:** `LogInformation` for normal flow, `LogError` for exceptions (future enhancement)

### 7.4 Validation

```
Request → Controller [FromBody] → MediatR Pipeline → ValidationBehavior → Handler
                                      │
                                      ├── Finds all IValidator<TRequest>
                                      ├── Runs ValidateAsync() on each
                                      └── Throws ValidationException on failure → 400 response
```

- **Library:** FluentValidation 12.1
- **Registration:** `AddValidatorsFromAssemblies(...)` auto-discovers all `AbstractValidator<T>`
- **Pipeline:** `ValidationBehavior` is registered as an open generic `IPipelineBehavior<,>`
- **Per-module:** Each module has its own validators. No shared validation logic across modules.

### 7.5 Configuration Management

| Source | Purpose | Example |
|--------|---------|---------|
| `appsettings.json` | Non-sensitive defaults | `MongoDb:ConnectionString`, `Jwt:Issuer` |
| `appsettings.Development.json` | Dev overrides | (empty — secrets go to user-secrets) |
| **User Secrets** | Sensitive values | `Jwt:SigningKey`, `RabbitMq:Username`, `RabbitMq:Password` |
| `JwtOptions` class | Strongly-typed config | `services.Configure<JwtOptions>(...)` |


## 8. Service Communication Patterns {#8}

### Communication Matrix

| From → To | Protocol | Mechanism | Where |
|-----------|----------|-----------|-------|
| Client → Api | HTTPS/JSON | REST Controllers | All `*Controller.cs` |
| Api → MongoDB | MongoDB Wire Protocol | Mongo.Entities `SaveAsync()` / `DB.Find()` | All handlers |
| Api → RabbitMQ | AMQP | MassTransit `IPublishEndpoint.Publish()` | `CreateTransactionHandler`, `RegisterHandler` |
| RabbitMQ → Host | AMQP | MassTransit Consumers | `FinTrack.Host/Program.cs` |
| Host → MongoDB | MongoDB Wire Protocol | Mongo.Entities | Consumer handlers |

### Synchronous (Request-Response)

```
Client ──POST /api/transactions──► Api.Controller ──ISender.Send()──► Handler ──SaveAsync()──► MongoDB
                                                                          │
                                                                    await Publish()
                                                                          │
                                                                          ▼
                                                                      RabbitMQ
```

- **Protocol:** HTTPS + JSON
- **Serialization:** ASP.NET Core model binding (`[FromBody]`)
- **Response:** `200 OK` with JSON body (`new { TransactionId = id }`)
- **Auth:** JWT Bearer token in `Authorization` header

### Asynchronous (Event-Driven)

```
Api.Handler ──Publish(TransactionCreatedEvent)──► RabbitMQ ──► Host.Consumer ──► Update Projections
                                                                                   │
                                                                                   ├── Dashboard: update snapshot
                                                                                   └── Budgets: update CurrentSpend
```

- **Broker:** RabbitMQ 3.x
- **Abstraction:** MassTransit (handles serialization, routing, retries)
- **Event naming:** Past-tense (`TransactionCreated`, `UserRegistered`)
- **Payload:** Immutable `record` in `FinTrack.Contracts`
- **Idempotency:** Consumers should be idempotent (not yet hardened)

### Module Isolation by Convention

| Rule | Why |
|------|-----|
| String foreign keys only | No module-to-module type dependencies |
| Integration events for side effects | Budgets react to `TransactionCreated` without referencing Transactions module |
| Dashboard is read-only | Never writes to other modules' collections |
| No sync cross-module calls | Prefer eventual consistency over distributed transactions |


## 9. .NET-Specific Architectural Patterns {#9}

### Host Model

| Host | Type | Builder | Startup |
|------|------|---------|---------|
| **Api** | `WebApplication` | `WebApplication.CreateBuilder(args)` | `await app.RunAsync()` |
| **Host** | `IHost` | `Host.CreateApplicationBuilder(args)` | `await host.RunAsync()` |

### Middleware Pipeline (Api)

```
Exception Handler → HTTPS Redirect → CORS → Authentication → Authorization → Controllers
```

Order matters:
1. **HTTPS first** — redirect before any processing
2. **CORS before Auth** — preflight requests must bypass auth
3. **Authentication before Authorization** — identity must be established
4. **Controllers last** — all middleware runs before routing

### Controller Pattern

```csharp
[ApiController]                     // Auto model validation + binding
[Route("api/{resource}")]          // Base route
[Authorize]                        // Applied to ALL actions unless overridden
public class XController : ControllerBase
{
    private readonly ISender _sender;   // MediatR — never business logic directly

    [HttpPost]                          // HTTP method
    public async Task<IActionResult> ActionName(  // Method name = endpoint name
        [FromBody] Command command,
        CancellationToken ct)
    {
        var result = await _sender.Send(command, ct);
        return Ok(result);
    }
}
```

### Dependency Injection

| Lifetime | Usage |
|----------|-------|
| `Singleton` | `SystemCurrentUser` (Host only — no request scope) |
| `Scoped` | `ICurrentUser` (Api — per HTTP request), `ISender`, `IPublishEndpoint` |
| `Transient` | Validators (via `AddValidatorsFromAssemblies`) |

### Assembly Scanning

MediatR, FluentValidation, and MassTransit all use assembly scanning — no manual handler registration:

```csharp
// MediatR discovers:
//   - All IRequestHandler<TRequest, TResponse>
//   - All IPipelineBehavior<TRequest, TResponse>
cfg.RegisterServicesFromAssemblies(module1Assembly, module2Assembly, ...);

// FluentValidation discovers:
//   - All AbstractValidator<T>
services.AddValidatorsFromAssemblies(moduleAssemblies);

// MassTransit discovers:
//   - All IConsumer<T> (Host only)
cfg.AddConsumers(moduleAssembly);
```


## 10. Implementation Patterns {#10}

### Controller Implementation Template

```csharp
[ApiController]
[Route("api/{resource}")]
[Authorize]
public class {Resource}Controller : ControllerBase
{
    private readonly ISender _sender;
    public {Resource}Controller(ISender sender) => _sender = sender;

    [HttpPost]
    public async Task<IActionResult> {ActionName}(
        [FromBody] {Action}Command command, CancellationToken ct)
    {
        var result = await _sender.Send(command, ct);
        return Ok(result);
    }
}
```

### Handler Implementation Template

```csharp
internal sealed class {Action}Handler : IRequestHandler<{Action}Command, string>
{
    private readonly ICurrentUser _currentUser;
    public {Action}Handler(ICurrentUser currentUser) => _currentUser = currentUser;

    public async Task<string> Handle({Action}Command request, CancellationToken ct)
    {
        var userId = _currentUser.UserId
            ?? throw new UnauthorizedAccessException();

        var entity = new Domain.{Entity}
        {
            UserId = userId,
            // map request fields
            CreatedBy = userId,
            CreateDate = DateTime.UtcNow
        };

        await entity.SaveAsync(cancellation: ct);
        return entity.ID;
    }
}
```

### Validator Implementation Template

```csharp
internal sealed class {Action}Validator : AbstractValidator<{Action}Command>
{
    public {Action}Validator()
    {
        RuleFor(x => x.Field).NotEmpty().MaximumLength(200);
        RuleFor(x => x.Amount).GreaterThan(0);
        RuleFor(x => x.Type).IsInEnum();
    }
}
```

### Domain Entity Template

```csharp
[Collection("{collectionName}")]
public class {Entity} : AuditableEntity
{
    public string UserId { get; set; } = string.Empty;
    // domain-specific fields
}
```

### Integration Event Template

```csharp
namespace FinTrack.Contracts.{Module};

public record {Entity}{Action}Event(
    string {Entity}Id,
    string UserId,
    // payload fields
    DateTime OccurredAt);
```

### Module DI Template

```csharp
namespace FinTrack.Modules.{Name};

public static class DependencyInjection
{
    public static IServiceCollection Add{Name}Module(this IServiceCollection services)
    {
        // Register module-specific services here
        return services;
    }
}
```


## 11. Testing Architecture {#11}

### Test Project Structure

```
tests/
├── FinTrack.Modules.Dashboard.Tests/
├── FinTrack.Modules.Transactions.Tests/
├── FinTrack.Modules.Categories.Tests/
├── FinTrack.Modules.Budgets.Tests/
├── FinTrack.Modules.Accounts.Tests/
└── FinTrack.Modules.Users.Tests/
```

### Test Stack

| Tool | Purpose |
|------|---------|
| **xUnit** | Test framework (`[Fact]`, `[Theory]`) |
| **FluentAssertions** | Readable assertions (`result.Should().Be(...)`) |
| **NSubstitute** | Mocking (`Substitute.For<ISender>()`) |
| **coverlet** | Code coverage |

### Test Strategy

| Level | What to Test | Example |
|-------|-------------|---------|
| **Unit — Handlers** | Business logic, orchestration | Mock `ICurrentUser`, assert correct entity saved |
| **Unit — Validators** | Validation rules | Pass valid/invalid commands, check errors |
| **Unit — Domain** | Entity invariants, BCrypt verify | `BCrypt.Verify(password, hash)` assertions |
| **Unit — Auth** | Token claims shape, refresh rotation | Decode JWT, check claims |
| **Integration** | API → MongoDB | `WebApplicationFactory` + Testcontainers (future) |

### Naming Convention

```
{MethodName}_When{Condition}_Returns{ExpectedResult}

Examples:
  Handle_WhenAmountIsNegative_ReturnsValidationError
  Handle_WhenUserNotAuthenticated_ThrowsUnauthorized
  Register_WhenEmailAlreadyExists_ThrowsInvalidOperation
```

### Test File Organization

```
Tests mirror the source feature folders:

src/FinTrack.Modules.Transactions/Features/CreateTransaction/
  CreateTransactionHandler.cs
  CreateTransactionValidator.cs

tests/FinTrack.Modules.Transactions.Tests/Features/CreateTransaction/
  CreateTransactionHandlerTests.cs
  CreateTransactionValidatorTests.cs
```


## 12. Deployment Architecture {#12}

### Deployment Topology

```
┌──────────────────────────────────────────┐
│           Docker Host                     │
│                                           │
│  ┌─────────────┐  ┌─────────────┐        │
│  │  mongo:7.0  │  │ rabbitmq:3  │        │
│  │  :27017     │  │ :5672:15672 │        │
│  └─────────────┘  └─────────────┘        │
│         ▲                ▲               │
│         │                │               │
│  ┌──────┴────────────────┴──────┐        │
│  │     FinTrack.Api (:5171)     │        │
│  │     FinTrack.Host            │        │
│  │     (run via dotnet CLI)     │        │
│  └──────────────────────────────┘        │
└──────────────────────────────────────────┘
```

### docker-compose.yml

```yaml
services:
  mongodb:
    image: mongo:7.0
    ports: ["27017:27017"]
    environment:
      MONGO_INITDB_DATABASE: FinTrackDb
    volumes: [mongo-data:/data/db]

  rabbitmq:
    image: rabbitmq:3-management
    ports: ["5672:5672", "15672:15672"]
```

### Configuration Environment Strategy

| Environment | SigningKey | RabbitMQ Credentials | MongoDB Connection |
|------------|-----------|---------------------|-------------------|
| **Development** | User Secrets | User Secrets | `appsettings.json` |
| **CI/Staging** | Env Var / GitHub Secret | Env Var | Env Var |
| **Production** | Azure Key Vault / AWS Secrets Manager | Vault | Vault |

### Runtime Dependencies

```
Api process needs:
  ✅ MongoDB running (connection string reachable)
  ✅ RabbitMQ running (AMQP port reachable)
  ❌ Host process (independent)

Host process needs:
  ✅ MongoDB running
  ✅ RabbitMQ running
  ❌ Api process (independent)
```


## 13. Extension & Evolution Patterns {#13}

### Adding a New Feature to an Existing Module

**Checklist:**

1. Create folder: `Features/{NewFeature}/`
2. Create files:
   - `{NewFeature}Command.cs` — `record : IRequest<TResponse>`
   - `{NewFeature}Handler.cs` — `internal sealed : IRequestHandler<TCommand, TResponse>`
   - `{NewFeature}Validator.cs` — `internal sealed : AbstractValidator<TCommand>`
3. Add action to module's `Controllers/{Module}Controller.cs`
4. MediatR + FluentValidation auto-discover via assembly scanning — **no DI registration needed**
5. If publishing events: inject `IPublishEndpoint` in handler, add event `record` to `Contracts`

### Adding a New Module

**Checklist:**

1. Create project: `dotnet new classlib -n FinTrack.Modules.{Name}`
2. Add `.csproj` references to `BuildingBlocks` + `Contracts`
3. Create `DependencyInjection.cs` with `Add{Name}Module()`
4. Add `Domain/` with entities inheriting `AuditableEntity`
5. Add `Controllers/{Name}Controller.cs` with `[ApiController]` + `[Authorize]`
6. Add feature slices under `Features/`
7. Register module in `Api/Program.cs`:
   ```csharp
   builder.Services.Add{Name}Module();
   ```
8. Register module assembly for MediatR scanning:
   ```csharp
   cfg.RegisterServicesFromAssemblies(typeof(FinTrack.Modules.{Name}.DependencyInjection).Assembly);
   ```
9. Add module assembly for FluentValidation (if module has validators)
10. Create test project: `tests/FinTrack.Modules.{Name}.Tests/`

### Adding a New Integration Event

1. Add `record` in `src/FinTrack.Contracts/{Module}/` — past-tense name
2. Publish in handler: `await _publishEndpoint.Publish(new XEvent(...), ct)`
3. If consumed: add `IConsumer<XEvent>` in consuming module's `EventHandlers/`
4. Register consumer in `Host/Program.cs`: `cfg.AddConsumers(moduleAssembly)`

### Extension Points

| Extension Point | How |
|----------------|-----|
| New validation rules | Add to existing `Validator` or create new one |
| New pipeline behavior | Implement `IPipelineBehavior<,>` → register in MediatR config |
| New MongoDB collection | Create entity class with `[Collection]` attribute |
| New auth policy | `options.AddPolicy("Admin", ...)` in `Program.cs` |
| New config section | Add to `appsettings.json` + create options class + `services.Configure<T>()` |


## 14. Architectural Pattern Examples {#14}

### Layer Separation: Controller → MediatR → Handler → Mongo

```csharp
// Controller (Api layer — thin, no business logic)
[ApiController, Route("api/transactions"), Authorize]
public class TransactionsController : ControllerBase
{
    private readonly ISender _sender;
    public TransactionsController(ISender sender) => _sender = sender;

    [HttpPost]
    public async Task<IActionResult> CreateTransaction(
        [FromBody] CreateTransactionCommand command, CancellationToken ct)
    {
        var id = await _sender.Send(command, ct);  // ← One line: dispatch
        return Ok(new { TransactionId = id });
    }
}

// Command (Data Transfer Object — immutable)
public record CreateTransactionCommand(
    string Title, decimal Amount, TransactionType Type,
    string AccountId, string CategoryId) : IRequest<string>;

// Handler (Business Logic — orchestration)
internal sealed class CreateTransactionHandler
    : IRequestHandler<CreateTransactionCommand, string>
{
    private readonly ICurrentUser _currentUser;
    private readonly IPublishEndpoint _publishEndpoint;

    public async Task<string> Handle(CreateTransactionCommand request, CancellationToken ct)
    {
        var userId = _currentUser.UserId
            ?? throw new UnauthorizedAccessException();

        var transaction = new Domain.Transaction
        {
            UserId = userId, Title = request.Title,
            Amount = request.Amount, Type = request.Type,
            AccountId = request.AccountId, CategoryId = request.CategoryId
        };

        await transaction.SaveAsync(cancellation: ct);            // ← Persist

        await _publishEndpoint.Publish(new TransactionCreatedEvent( // ← Notify
            transaction.ID, userId, request.AccountId,
            request.CategoryId, request.Title, request.Amount,
            request.Type.ToString(), DateTime.UtcNow), ct);

        return transaction.ID;
    }
}
```

### Event-Driven: Register → Publish → Categories Seed

```csharp
// Producer (Users module)
await _publishEndpoint.Publish(new UserRegisteredEvent(
    user.ID, user.Email, DateTime.UtcNow), ct);

// Consumer (Categories module — in Host)
public class SeedDefaultCategoriesConsumer : IConsumer<UserRegisteredEvent>
{
    public async Task Consume(ConsumeContext<UserRegisteredEvent> context)
    {
        var msg = context.Message;
        // Create default categories for new user
        await new Category { UserId = msg.UserId, Name = "Salary", Type = CategoryType.Income }
            .SaveAsync();
        await new Category { UserId = msg.UserId, Name = "Groceries", Type = CategoryType.Expense }
            .SaveAsync();
    }
}
```

### Validation Pipeline: FluentValidation + MediatR Behavior

```csharp
// Validator
internal sealed class CreateTransactionValidator
    : AbstractValidator<CreateTransactionCommand>
{
    public CreateTransactionValidator()
    {
        RuleFor(x => x.Title).NotEmpty().MaximumLength(200);
        RuleFor(x => x.Amount).GreaterThan(0);
        RuleFor(x => x.Type).IsInEnum();
    }
}

// Behavior (runs automatically for every IRequest<T>)
public class ValidationBehavior<TRequest, TResponse>
    : IPipelineBehavior<TRequest, TResponse>
{
    public async Task<TResponse> Handle(TRequest request,
        RequestHandlerDelegate<TResponse> next, CancellationToken ct)
    {
        var failures = _validators
            .Select(v => v.Validate(request))
            .SelectMany(r => r.Errors)
            .Where(f => f is not null).ToList();

        if (failures.Count != 0)
            throw new ValidationException(failures);

        return await next();  // ✅ All valid → proceed to handler
    }
}
```


## 15. Architectural Decision Records (ADR) {#15}

### ADR-001: Modular Monolith over Microservices

| Aspect | Detail |
|--------|--------|
| **Context** | FinTrack is a personal finance app. Early-stage, small team. |
| **Decision** | Use Modular Monolith — 6 class libraries in-process, not 6 microservices. |
| **Alternatives** | Microservices (too complex), single-project monolith (no boundaries). |
| **Consequences** | ✅ Simple deployment (2 processes). ✅ Fast development. ⚠️ Must enforce module boundaries by convention, not by network. |

### ADR-002: Controllers over Minimal APIs

| Aspect | Detail |
|--------|--------|
| **Context** | ASP.NET Core supports both Minimal APIs and Controllers. |
| **Decision** | Use Controllers (`[ApiController]`) — method name = endpoint name. |
| **Alternatives** | Minimal APIs (less ceremony, but method names are anonymous lambdas). |
| **Consequences** | ✅ Clean method naming. ✅ Familiar pattern. ✅ Swagger auto-documentation. |

### ADR-003: Mongo.Entities over Raw MongoDB.Driver

| Aspect | Detail |
|--------|--------|
| **Context** | Need CRUD operations on MongoDB. |
| **Decision** | Use Mongo.Entities for 90% of operations; fall back to MongoDB.Driver for complex queries. |
| **Alternatives** | Raw driver only (verbose), EF Core with MongoDB provider (immature). |
| **Consequences** | ✅ `SaveAsync()`, `DB.Find<T>()` — concise. ⚠️ Aggregation pipelines still need raw driver. |

### ADR-004: MassTransit over Raw RabbitMQ Client

| Aspect | Detail |
|--------|--------|
| **Context** | Cross-module communication needs a message broker. |
| **Decision** | MassTransit abstraction over RabbitMQ. |
| **Alternatives** | Raw `RabbitMQ.Client` (boilerplate), Azure Service Bus (cloud lock-in). |
| **Consequences** | ✅ Serialization, routing, retries handled. ✅ Easy to swap transport later. |

### ADR-005: JWT with Fallback Policy (Deny by Default)

| Aspect | Detail |
|--------|--------|
| **Context** | All business endpoints must be authenticated. |
| **Decision** | `options.FallbackPolicy = options.DefaultPolicy` — every endpoint requires auth unless `[AllowAnonymous]`. |
| **Alternatives** | Per-endpoint `[Authorize]` (error-prone — easy to forget). |
| **Consequences** | ✅ Secure by default. ✅ New endpoints auto-protected. |

### ADR-006: No Outbox Pattern (Yet)

| Aspect | Detail |
|--------|--------|
| **Context** | Eventual consistency between modules via RabbitMQ. |
| **Decision** | Start without transactional outbox. Publish after successful save. |
| **Alternatives** | Outbox pattern (adds complexity — deferred to when projections must not miss events). |
| **Consequences** | ⚠️ Possible event loss if process crashes between save and publish. Acceptable for MVP. |


## 16. Architecture Governance {#16}

### Automated Checks

| Check | Tool | Status |
|-------|------|--------|
| **No cross-module references** | `.csproj` analysis — no `FinTrack.Modules.*` → `FinTrack.Modules.*` ProjectReferences | ✅ Enforced at build |
| **Build succeeds** | `dotnet build FinTrack.slnx` | ✅ CI-ready |
| **Tests pass** | `dotnet test FinTrack.slnx` | Ready |
| **Code coverage** | coverlet + `dotnet test --collect:"XPlat Code Coverage"` | Future |

### Manual Review Checklist

When reviewing a PR, verify:

- [ ] New features are in `Features/{UseCase}/` folders, not in a shared Application layer
- [ ] No business logic in Controllers — only `_sender.Send(command, ct)`
- [ ] Handlers are `internal sealed`
- [ ] Validators exist for every command with input
- [ ] New entities inherit `AuditableEntity` and use `[Collection]` attribute
- [ ] No cross-module `ProjectReference` in `.csproj`
- [ ] Integration events are `record` types in `FinTrack.Contracts`
- [ ] Cross-module IDs are strings, not object references
- [ ] `UserId` is scoped in every financial entity
- [ ] Auth: `[Authorize]` on controller unless explicitly anonymous

### Documentation

| Document | Location | Purpose |
|----------|----------|---------|
| Architecture conventions | `docs/architecture-modular-monolith-vsa.md` | Rules for module boundaries, VSA, auth |
| System instructions | `docs/system-instructions.md` | How to run, configure, debug |
| Learning path | `docs/learning-path.md` | Knowledge prerequisites |
| **This blueprint** | `docs/architecture-blueprint.ipynb` | Reference for architectural consistency |


## 17. Blueprint for New Development {#17}

### Development Workflow

```
1. Pick a module (or create one)
   └── Refer to Module Boundaries table (§4) for ownership

2. Create feature folder
   └── src/FinTrack.Modules.{Name}/Features/{UseCase}/

3. Add files in order:
   a. {UseCase}Command.cs    — record : IRequest<TResponse>
   b. {UseCase}Validator.cs  — AbstractValidator<TCommand>
   c. {UseCase}Handler.cs    — IRequestHandler<TCommand, TResponse>

4. Add controller action
   └── Controllers/{Name}Controller.cs → new [HttpPost] method

5. If cross-module side effects:
   a. Add event record in Contracts/
   b. Inject IPublishEndpoint in handler
   c. (Optional) Add consumer in consuming module

6. Build & test
   └── dotnet build && dotnet test
```

### File Creation Template

**New command:**
```csharp
namespace FinTrack.Modules.{Name}.Features.{UseCase};

public record {UseCase}Command(/* fields */) : IRequest<string>;
```

**New validator:**
```csharp
namespace FinTrack.Modules.{Name}.Features.{UseCase};

internal sealed class {UseCase}Validator : AbstractValidator<{UseCase}Command>
{
    public {UseCase}Validator()
    {
        RuleFor(x => x.Field).NotEmpty();
    }
}
```

**New handler:**
```csharp
namespace FinTrack.Modules.{Name}.Features.{UseCase};

internal sealed class {UseCase}Handler : IRequestHandler<{UseCase}Command, string>
{
    private readonly ICurrentUser _currentUser;

    public {UseCase}Handler(ICurrentUser currentUser) => _currentUser = currentUser;

    public async Task<string> Handle({UseCase}Command request, CancellationToken ct)
    {
        var userId = _currentUser.UserId
            ?? throw new UnauthorizedAccessException();
        // business logic
    }
}
```

**New controller action:**
```csharp
[HttpPost]
public async Task<IActionResult> {UseCase}(
    [FromBody] {UseCase}Command command, CancellationToken ct)
{
    var result = await _sender.Send(command, ct);
    return Ok(result);
}
```

### Common Pitfalls

| ❌ Don't | ✅ Do |
|---------|------|
| Add business logic to Controllers | Keep controllers thin — only `_sender.Send()` |
| Reference another module's entity type | Use string IDs + integration events |
| Create a shared `ApplicationService` class | Put logic in feature-specific handlers |
| Forget `[Authorize]` on new controller | Auth is fallback — but be explicit |
| Skip the validator | Every command with user input needs a validator |
| Hardcode secrets in `appsettings.json` | Use User Secrets / env vars |
| Add cross-module ProjectReference | Communication via Contracts or RabbitMQ only |
| Store plaintext passwords | Always use BCrypt (`HashPassword` / `Verify`) |

---

## 📅 Blueprint Maintenance

- **Generated:** 2026-07-26
- **Update cadence:** After every major feature or module addition
- **Version:** 1.0
- **Related docs:** `architecture-modular-monolith-vsa.md`, `system-instructions.md`, `learning-path.md`
